# 04 — Bronze: IMF IMTS bilateral trade data

**Architecture:** Databricks Free Edition serverless cannot reach `api.imf.org` (DNS blocked).  
Data is extracted locally on Mac and uploaded to a Databricks Volume before this notebook runs.

The **IMTS** dataflow (International Merchandise Trade Statistics) replaces the retired `dataservices.imf.org/DOTS` service (decommissioned November 2025).

## Steps before running this notebook

1. **Extract locally on Mac** (no API key needed):
   ```bash
   pip install -r extraction/requirements.txt
   python3 extraction/extract/imf_dots_extract.py \
     --all-cemac-ecowas --start-year 1990 --end-year 2024 \
     --out data/raw/imts/cemac_ecowas_imts_1990_2024.jsonl
   ```

2. **Upload JSONL to Volume**:
   ```bash
   databricks fs cp \
     data/raw/imts/cemac_ecowas_imts_1990_2024.jsonl \
     dbfs:/Volumes/cemac_ecowas_aes_trade/bronze/raw_landing/imts/cemac_ecowas_imts_1990_2024.jsonl
   ```

3. **Run this notebook** — reads JSONL, writes `bronze.imts_raw`.

**Indicators:** `XG_FOB_USD` (exports FOB) · `MG_CIF_USD` (imports CIF).  
**Country codes:** ISO 3-letter (CMR, W00 for World aggregate, etc.)

In [ ]:
# Cell 1 — Configuration
spark.sql("USE CATALOG cemac_ecowas_aes_trade")
spark.sql("USE SCHEMA bronze")

VOLUME_PATH = "/Volumes/cemac_ecowas_aes_trade/bronze/raw_landing/imts/"

CEMAC_ISO3  = ["CMR", "CAF", "TCD", "COG", "GNQ", "GAB"]
ECOWAS_ISO3 = ["BEN", "BFA", "CPV", "CIV", "GMB", "GHA",
               "GIN", "GNB", "LBR", "MLI", "NER", "NGA",
               "SEN", "SLE", "TGO"]
ALL_REPORTERS = CEMAC_ISO3 + ECOWAS_ISO3

YEARS = list(range(1990, 2025))

print(f"Catalog:   cemac_ecowas_aes_trade")
print(f"Schema:    bronze")
print(f"Volume:    {VOLUME_PATH}")
print(f"Reporters: {len(ALL_REPORTERS)} countries (ISO3 codes)")
print(f"Years:     {YEARS[0]}–{YEARS[-1]} ({len(YEARS)} years)")

In [ ]:
# Cell 3 — Country label lookup (hardcoded — Databricks cannot reach api.imf.org)
COUNTRY_LABELS = {
    # CEMAC
    "CMR": "Cameroon",
    "CAF": "Central African Republic",
    "TCD": "Chad",
    "COG": "Congo",
    "GNQ": "Equatorial Guinea",
    "GAB": "Gabon",
    # ECOWAS
    "BEN": "Benin",
    "BFA": "Burkina Faso",
    "CPV": "Cabo Verde",
    "CIV": "Côte d'Ivoire",
    "GMB": "Gambia",
    "GHA": "Ghana",
    "GIN": "Guinea",
    "GNB": "Guinea-Bissau",
    "LBR": "Liberia",
    "MLI": "Mali",
    "NER": "Niger",
    "NGA": "Nigeria",
    "SEN": "Senegal",
    "SLE": "Sierra Leone",
    "TGO": "Togo",
    # World aggregate
    "W00": "World",
}

missing_reporters = [code for code in ALL_REPORTERS if code not in COUNTRY_LABELS]
if missing_reporters:
    raise ValueError(f"Missing reporter code(s) in lookup: {missing_reporters}")

print(f"Loaded {len(COUNTRY_LABELS)} country/partner labels")
print("Project reporters:")
for code in ALL_REPORTERS:
    print(f"  {code}: {COUNTRY_LABELS[code]}")


In [ ]:
# Cell 2 — List JSONL files in the Volume
files = [f for f in dbutils.fs.ls(VOLUME_PATH) if f.name.endswith(".jsonl")]

if not files:
    raise FileNotFoundError(
        f"No .jsonl files found at {VOLUME_PATH}. "
        "Run imf_dots_extract.py locally and upload the output first."
    )

print(f"Found {len(files)} JSONL file(s):")
for f in files:
    print(f"  {f.name}  ({f.size:,} bytes)")

In [ ]:
# Cell 3 — Load observations from JSONL envelopes
import json
from datetime import datetime, timezone

all_observations = []
loaded_at = datetime.now(timezone.utc)

for file_info in files:
    for row in spark.read.text(file_info.path).collect():
        line = row.value.strip()
        if not line:
            continue
        envelope = json.loads(line)
        for obs in envelope.get("payload", []):
            all_observations.append(obs)

print(f"Total raw observations loaded: {len(all_observations):,}")


In [ ]:
# Cell 4 — Inspect a sample observation
if all_observations:
    sample = all_observations[0]
    print("Sample observation (first row):")
    for key, value in sample.items():
        print(f"  {key}: {value}")
    print(f"
Total fields per observation: {len(sample)}")
else:
    print("No observations loaded. Check Cell 3 output.")

In [ ]:
# Cell 5 — Build Spark DataFrame
from pyspark.sql import Row

rows = []
for obs in all_observations:
    year = obs.get("year")
    if not year or int(year) < 1990 or int(year) > 2024:
        continue

    rows.append(Row(
        reporter_iso3    = obs.get("reporter_iso3", ""),
        counterpart_iso3 = obs.get("counterpart_iso3", ""),
        indicator        = obs.get("indicator", ""),  # XG_FOB_USD or MG_CIF_USD
        year             = int(year),
        value_usd        = float(obs["value_usd"]) if obs.get("value_usd") is not None else None,
        source           = "imf_imts",
        loaded_at        = loaded_at,
    ))

df = spark.createDataFrame(rows)
print(f"DataFrame: {df.count():,} rows")
df.printSchema()
df.show(5, truncate=False)

In [ ]:
# Cell 6 — Write to bronze.imts_raw
spark.sql("DROP TABLE IF EXISTS bronze.imts_raw")

(df.write
   .mode("overwrite")
   .option("overwriteSchema", "true")
   .saveAsTable("bronze.imts_raw"))

print("Write complete.")

In [ ]:
# Cell 7 — Verify coverage
print("Overall coverage:")
spark.sql("""
    SELECT
        indicator,
        COUNT(*)                               AS rows,
        COUNT(DISTINCT reporter_iso3)          AS reporters,
        COUNT(DISTINCT counterpart_iso3)       AS counterparts,
        COUNT(DISTINCT year)                   AS years,
        MIN(year)                              AS earliest,
        MAX(year)                              AS latest,
        ROUND(SUM(value_usd) / 1e12, 2)       AS total_trillions_usd
    FROM bronze.imts_raw
    GROUP BY indicator
    ORDER BY indicator
""").show(truncate=False)

# World total = sum across all bilateral counterparts
print("Cameroon total exports by year (sum across all counterparts):")
spark.sql("""
    SELECT year,
           ROUND(SUM(value_usd) / 1e9, 2) AS exports_billions_usd
    FROM   bronze.imts_raw
    WHERE  reporter_iso3 = 'CMR'
      AND  indicator     = 'XG_FOB_USD'
    GROUP BY year
    ORDER BY year
""").show(40, truncate=False)
